In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split


In [9]:
data = np.load('kanji_data.npz', allow_pickle=True)
X = data['images']
y = data['labels']
class_list = data['class_list']
print(f"Label {y[0]} -> {class_list[y[0]]} -> {chr(int(class_list[y[0]], 16))}")

Label 0 -> 0x3042 -> あ


In [10]:
# normalize pixel values from [0, 255] to [0, 1]
X = X.astype('float32') / 255.0

# add channel dim
X = X[:, np.newaxis, :, :]

# split train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [7]:
# convert to tensors
X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train).long()
X_val = torch.from_numpy(X_val)
y_val = torch.from_numpy(y_val).long()
X_test = torch.from_numpy(X_test)
y_test = torch.from_numpy(y_test).long()

from torch.utils.data import TensorDataset, DataLoader

# batch
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
val_data = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=2)
test_data = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2)


In [13]:
image, label = train_data[0]
print(image.size())
print(len(class_list))

torch.Size([1, 127, 128])
3036


In [ ]:
class NeuralNet(nn.Module):

    def __init__ (self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 12, 5) # (12, 123, 124)
        self.pool = nn.MaxPool2d(2, 2) # (12, 61, 62)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 57, 58) -> apply pool -> (24, 28, 29) -> flatten (24 * 28 * 29)

        self.fc1 = nn.Linear(24 * 28 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3036) # 3036 kanji/character classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(x.fcl(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss =+ loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')